# Running the Snakemake Pipeline

This tutorial covers how to use NeuRodent's Snakemake pipeline for large-scale, reproducible EEG analysis workflows.

## What is Snakemake?

[Snakemake](https://snakemake.readthedocs.io/) is a workflow management system that creates reproducible, scalable data analyses. Key features:

- **Automatic dependency tracking**: Only recomputes what's needed when data or parameters change
- **Parallel execution**: Processes multiple samples simultaneously
- **Fault tolerance**: Resumes from where it left off after failures
- **Resource management**: Specify memory, CPU, and time requirements for each step
- **Cluster integration**: Seamlessly scales from laptops to HPC clusters

## Why Use the Snakemake Pipeline?

For experiments with many animals and days of recordings, manual analysis becomes impractical. The NeuRodent pipeline automates:

1. Raw EEG data → Windowed Analysis Results (WARs)
2. Quality filtering and artifact removal
3. Diagnostic figure generation
4. Statistical analyses and publication-ready plots
5. All intermediate processing steps


## Installation

Ensure you have NeuRodent and Snakemake installed:

```bash
# Install both with uv
uv add neurodent snakemake

# Or with pip
pip install neurodent snakemake
```


## Pipeline Overview

The pipeline processes data through multiple stages:

```
Raw EEG Data
    ↓
WAR Generation (windowed features)
    ↓
Quality Filtering (remove bad days/genotypes)
    ↓
Standardization (channel ordering)
    ↓
Fragment Filtering (temporal artifacts)
    ↓
Channel Filtering (spatial artifacts)
    ↓
WAR Flattening (aggregate across time)
    ↓
Zeitgeber Analysis (circadian features)
    ↓
Statistical Figures & Plots
```

Each stage produces intermediate files that are automatically cached. If you modify parameters or data, Snakemake only reruns affected steps.


## Configuration

The pipeline requires two configuration files in the `config/` directory:

### 1. `config/config.yaml` - Analysis Parameters

This file defines paths, analysis settings, and computational resources:

```yaml
# Dataset selection — dataset configs live in config/datasets/{name}.yaml
active_dataset: "sox5_bin"

# Core directories
temp_directory: "/path/to/temp/storage"

# Sample configuration
samples:
  quality_filter:
    exclude_unknown_genotypes: true
    exclude_bad_animaldays: true

# Analysis parameters
# Note: pattern and lro_kwargs.mode come from the dataset-specific config
analysis:
  war_generation:
    skip_sessions: ["bad"]
    lro_kwargs:
      multiprocess_mode: "dask"
  
  fragment_filter_config:
    logrms_range:
      z_range: 3
    high_beta:
      max_beta_prop: 0.4
  
  channel_filter_config:
    lof:
      reject_lof_threshold: 2.5
      min_valid_channels: 3

# Resource requirements (adjust for your system)
cluster:
  war_generation:
    time: "12h"
    mem_mb: 80_000
    threads: 4
```

Key parameters:
- **Paths**: Set `temp_directory` for your system; `data_root` is defined in `samples.json`
- **Dataset**: Set `active_dataset` to select a dataset config from `config/datasets/`
- **Analysis**: Configure filtering thresholds, detection parameters, etc.
- **Resources**: Memory, time, CPUs and retries per rule, used for cluster scheduling. Rules that read
  WARs size their memory from their inputs; see Running on HPC Clusters.


### 2. `config/samples.json` - Sample Metadata

This file defines which animals to process and their experimental metadata:

```json
{
  "data_root": "/path/to/raw/eeg/data",
  "animals": [
    {"id": "M1", "genotype": "WT", "sex": "M"},
    {"id": "M2", "genotype": "Het", "sex": "M"},
    {"id": "F1", "genotype": "WT", "sex": "F"},
    {"id": "F2", "genotype": "Mut", "sex": "F",
     "bad_channels": ["LHip", "RHip"]}
  ],
  "channels": {
    "LMot": ["LMot", "EEG E1-REF1"],
    "RMot": ["RMot", "EEG E2-REF1"],
    "LAud": ["LAud"], "RAud": ["RAud"],
    "LVis": ["LVis"], "RVis": ["RVis"]
  }
}
```

Key sections:
- **data_root**: Root path containing the raw data directories
- **animals**: List of animal entries with `id`, `genotype`, `sex`, and optional per-animal `bad_channels`, `pattern`, or `lro_kwargs` overrides
- **channels**: Maps each canonical channel abbreviation (e.g. `LMot`) to the **exact** raw channel names that appear in the data. Resolution is an exact lookup against these names — labels are never inferred.


## Running the Pipeline

### Basic Execution

Run the pipeline on your local machine or compute node:

```bash
# Dry run to preview what will execute
snakemake --dry-run

# Execute with 4 parallel jobs
snakemake --cores 4

# Run a specific analysis stage
snakemake --cores all results/wars_quality_filtered/

# Force rerun of a specific rule and downstream steps
snakemake --cores all --forcerun war_generation
```

### Useful Commands

```bash
# Visualize the workflow
snakemake --dag | dot -Tpng > dag.png
snakemake --rulegraph | dot -Tpng > rulegraph.png

# Unlock directory after crash
snakemake --unlock

# Clean up specific outputs to rerun
snakemake --delete-all-output

# Show which files are missing
snakemake --summary
```

### Monitoring Progress

Each job writes its prints and logging to `logs/<rule>/<animal>.out`, and its warnings and
tracebacks to `logs/<rule>/<animal>.err`. Rules that run once, such as war_zeitgeber, use the rule's
name in place of the animal. Snakemake deletes these two files before each attempt of a job, so they
hold the current attempt only. Every attempt is also appended to `logs/<rule>/<animal>.attempts.out`
and `.attempts.err`, after a line that names the attempt, its SLURM job and node, and its start time.
Snakemake never deletes these. They also hold the output of child processes such as Dask workers,
which the `.out` and `.err` files do not.

```bash
# Follow a running job
tail -f logs/war_generation/<animal>.out

# List every attempt of a job, retries included
grep -n '^=== ' logs/war_generation/<animal>.attempts.err

# Check results directory
ls -lh results/
```

## Running on HPC Clusters

For large-scale analyses, you can submit pipeline jobs to an HPC cluster scheduler. Snakemake supports multiple schedulers including SLURM, SGE, PBS, LSF, and others.

### General Cluster Execution

Snakemake can interface with any job scheduler using profiles. The key is to:

1. Define resource requirements in `config.yaml` (time, memory, threads)
2. Create a cluster profile for your scheduler
3. Submit the pipeline using the profile

Snakemake will automatically:
- Submit each rule as a separate job
- Manage job dependencies
- Monitor job status
- Resubmit failed jobs, up to the retries of the rule's cluster block. Every rule sets retries, which
  overrides the profile's `restart-times`, so `restart-times` has no effect

### Example: SLURM Scheduler

Here's how to set up Snakemake with SLURM (adapt for your scheduler):

```bash
# Create a SLURM profile (one-time setup)
mkdir -p ~/.config/snakemake/slurm

cat > ~/.config/snakemake/slurm/config.yaml << 'EOF'
cluster: "sbatch --time={resources.time} --mem={resources.mem_mb} --cpus-per-task={threads} --job-name={rule} --output=logs/{rule}/slurm_%j.out"
jobs: 100
EOF

# Submit pipeline
snakemake --profile slurm
```

The pipeline reads resource requirements from your `config.yaml`:

```yaml
cluster:
  war_generation:
    time: "12h"
    mem_mb: 80_000
    threads: 4
```

And Snakemake translates these to scheduler-specific flags. A rule that reads WARs asks for more
than its `mem_mb` when the parquet footers of its input WARs call for it (`mem_bytes_per_window` in
its block), so a dry run shows its memory as TBD until those WARs exist. A
`--set-resources rule:mem_mb=...` on the command line replaces that computation, including the
doubling on retry. `workflow/schemas/config.schema.yaml` lists the keys each block takes.


### Monitoring Cluster Jobs

For SLURM:
```bash
# Check job queue
squeue -u $USER

# View job details
sacct --format=JobID,JobName,State,Elapsed,MaxRSS

# Follow a running job
tail -f logs/war_generation/<animal>.out
```

When a job is retried, the failed attempt's output, Dask worker messages included, stays in
`logs/war_generation/<animal>.attempts.out` and `.attempts.err`. The SLURM job's own output file
(`logs/war_generation/slurm_<jobid>.out` with the profile above) holds only Snakemake's messages and
two lines naming those files.

For SGE:
```bash
# Check job queue
qstat -u $USER

# View job details
qacct -j <job_id>
```

For PBS:
```bash
# Check job queue
qstat -u $USER

# View job details
qstat -f <job_id>
```

### Other Schedulers

Snakemake supports many schedulers. See the [Snakemake cluster documentation](https://snakemake.readthedocs.io/en/stable/executing/cluster.html) for:
- **SGE** (Sun Grid Engine)
- **PBS/Torque**
- **LSF** (Load Sharing Facility)
- **Cloud executors** (Google Cloud, AWS, Azure)

The general pattern is the same: create a profile with your scheduler's submission command and resource mappings.


## Pipeline Outputs

The pipeline creates a structured results directory:

```
results/
├── wars_quality_filtered/       # Initial WARs with quality filtering applied
├── wars_standardized/            # Channel-reordered WARs
├── wars_fragment_filtered/       # After temporal artifact removal
├── wars_flattened_manual/        # Aggregated with manual channel filtering
├── wars_flattened_lof/           # Aggregated with LOF channel filtering
├── diagnostic_figures/           # Per-animal QC plots
│   └── {animal}/
│       ├── unfiltered/           # Pre-filtering diagnostics
│       └── filtered/             # Post-filtering diagnostics
├── wars_zeitgeber/               # Circadian time-aligned features
├── zeitgeber_plots/              # Temporal heatmaps
├── relfreq_plots/                # Feature distributions
├── ep_figures/                   # Statistical group comparisons
├── ep_heatmaps/                  # Connectivity matrices
├── lof_evaluation/               # LOF threshold optimization
└── filtering_comparison_plots/   # Manual vs LOF comparison

logs/
└── {rule}/
    ├── {animal}.out, {animal}.err                    # Current attempt of each job
    └── {animal}.attempts.out, {animal}.attempts.err  # Every attempt, with child process output
```

Key outputs:
- **`.pkl` files**: Python pickled WARs with computed features
- **`.json` files**: Metadata and analysis parameters
- **Figures**: PNG/PDF diagnostic and publication plots
- **CSV files**: Exportable tabular data


## Troubleshooting & Best Practices

### Common Issues

**Pipeline is locked:**
```bash
snakemake --unlock
```

**Need to rerun everything:**
```bash
snakemake --cores all --forceall
```

**Jobs failing due to insufficient memory:**
Increase memory in `config.yaml` under `cluster` section for the failing rule.

**Want to test changes without full pipeline:**
```bash
# Run just one rule
snakemake --cores all war_generation

# Or target specific output
snakemake --cores all results/wars_quality_filtered/animal1/
```

### Best Practices

1. **Always dry-run first** to preview execution:
   ```bash
   snakemake --dry-run
   ```

2. **Test on small subset** before full dataset:
   - Temporarily modify `samples.json` to include just 1-2 animals
   - Run locally to verify parameters
   - Expand to full dataset once validated

3. **Monitor resource usage**:
   - Check logs for memory errors
   - Adjust `cluster` resources in `config.yaml` as needed
   - Most rules should complete within their time limits

4. **Keep intermediate files**:
   - Don't delete results directories - Snakemake uses them for caching
   - If rerunning from scratch, use `--forceall` instead of deleting

5. **Version control your configs**:
   - Track `config/config.yaml` and `config/samples.json` in git
   - Document parameter changes for reproducibility

## Next Steps

Now that you understand the Snakemake pipeline:

- **Explore outputs**: Review diagnostic figures in `results/diagnostic_figures/` to assess data quality
- **Customize analysis**: Modify parameters in `config/config.yaml` for your experimental needs
- **Interactive analysis**: Load `.pkl` WAR files in Python for custom analyses
- **Learn more**: Check out other tutorials:
  - [Windowed Analysis](windowed_analysis.ipynb) - Understanding feature extraction
  - [Visualization](visualization.ipynb) - Creating custom plots
  - [Basic Usage](../quickstart/basic_usage.ipynb) - Interactive Python workflow

## Additional Resources

- [Snakemake Documentation](https://snakemake.readthedocs.io/)
- [Snakemake Cluster Execution](https://snakemake.readthedocs.io/en/stable/executing/cluster.html)
- [Snakemake Profiles](https://github.com/Snakemake-Profiles) - Community-maintained scheduler profiles
